# Self-RAG — Teach an LLM to Know When to Look Things Up
## Retrieve or Not? Let the LLM Decide
⏱ ~15 min

**Self-RAG** (Asai et al., 2023) replaces the *always-retrieve* assumption of standard RAG with three LLM-powered decision gates. The model asks itself: *Should I even retrieve? Are these docs relevant? Is my answer actually supported?* Each gate emits a structured **reflection token** — `ISRET`, `ISREL`, `ISSUP` — that controls the execution path.

By the end of this session you will understand *why* always-on retrieval is wasteful, *how* structured LLM outputs act as conditional routing logic, and *how* to wire all three gates into a LangGraph workflow.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Standard RAG vs CRAG vs Self-RAG |
| 2 | **Reflection Tokens** — What ISRET / ISREL / ISSUP mean |
| 3 | **Knowledge Base** — Build the vector store |
| 4 | **Gate 1 — classify** — Should I retrieve at all? |
| 5 | **Gate 2 — grade_docs** — Are these docs relevant? |
| 6 | **Gate 3 — check_support** — Is my answer grounded? |
| 7 | **Full Graph** — Wire all three gates with LangGraph |
| 8 | **Run & Observe** — End-to-end demo with trace output |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing `requirements.txt` dependencies
- An `OPENAI_API_KEY` in `.env` (local) or Colab Secrets

### Key References
> Asai, A., Wu, Z., Wang, Y., Sil, A., & Hajishirzi, H. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection.* NeurIPS 2023. https://arxiv.org/abs/2310.11511
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
> LangGraph docs: https://langchain-ai.github.io/langgraph/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "langgraph",
            "chromadb",
            "pydantic",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ────────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), name = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid.\n"
    "Local: add OPENAI_API_KEY=sk-... to .env\n"
    "Colab:  Secrets panel → Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True  ({key[:8]}...)")

---

## Part 1 — Concepts: Standard RAG vs CRAG vs Self-RAG

---

### The Problem with Always-On Retrieval

Standard RAG **always** retrieves — even when the answer is trivial common knowledge ("What is 2 + 2?") or a greeting ("Hello!"). This wastes latency and token budget, and sometimes retrieves irrelevant chunks that actively mislead the model.

**Self-RAG** addresses this by teaching the LLM to use *reflection tokens* — special signals that gate each phase of the pipeline.

---

### Three Approaches Compared

| Approach | Retrieval trigger | Document grading | Groundedness check | Re-try logic |
|----------|------------------|------------------|--------------------|---------------|
| **Standard RAG** | Always | None | None | None |
| **CRAG** (Corrective RAG) | Always | Yes — rewrite query if bad | None | Yes — web fallback |
| **Self-RAG** | LLM decides per query | Yes — filter irrelevant | Yes — isSupported? | Possible (retry gate) |

Self-RAG is the most adaptive: it skips retrieval entirely for queries the model can answer from weights, grades each retrieved document before generation, and then verifies the answer is actually grounded in what was retrieved.

---

### Reflection Token Types

The original Self-RAG paper defines four reflection tokens. This workshop implements three of them using a `BoolDecision` Pydantic model as a drop-in proxy:

| Token | Name | Question asked | When emitted |
|-------|------|----------------|---------------|
| `[Retrieve]` / `ISRET` | **Retrieval trigger** | "Should I retrieve for this query?" | Before any retrieval |
| `[IsREL]` | **Relevance** | "Is this document relevant to the query?" | After retrieval, per document |
| `[IsSUP]` | **Support** | "Is the answer supported by the context?" | After generation |
| `[IsUSE]` | **Utility** | "Is this response useful to the user?" | After generation (optional) |

> In the original paper these tokens are embedded in the LLM output vocabulary through fine-tuning. We approximate them with structured LLM calls (`with_structured_output`) — same logic, no fine-tuning required.

---

## Part 2 — The Self-RAG Decision Graph

---

```
  User query
      │
      ▼
┌─────────────┐
│   classify  │  Gate 1: ISRET — Should I retrieve?
└──────┬──────┘
       │
  ┌────┴─────┐
  │          │
 True      False
  │          │
  ▼          │
┌──────────┐ │
│ retrieve │ │   Fetch top-k docs from vector store
└────┬─────┘ │
     │       │
     ▼       │
┌───────────┐│
│ grade_docs││  Gate 2: ISREL — per-document relevance filter
└────┬──────┘│
     │       │
     └───────┘
          │
          ▼
    ┌──────────┐
    │ generate │  LLM generates answer (with or without context)
    └────┬─────┘
         │
         ▼
  ┌─────────────┐
  │check_support│  Gate 3: ISSUP — Is the answer grounded?
  └──────┬──────┘
         │
         ▼
       [END]
```

> **Key insight:** When `should_retrieve = False`, the graph short-circuits directly to `generate`, skipping the retrieval + grading cost entirely. Gate 3 still runs — even answers from model weights should be verified for groundedness when context exists.

---

## Part 3 — Build the Knowledge Base

---

We build a small in-memory ChromaDB vector store. In a real pipeline this would be replaced by a persistent store loaded at startup.

The corpus deliberately mixes:
- **Highly relevant** docs (Self-RAG, LangGraph, CRAG topics)
- **General knowledge** (Eiffel Tower) — so Gate 1 can decide to skip retrieval for that class of question
- **Adjacent but not identical** docs (Adaptive RAG) — so Gate 2 can exercise its relevance filter

In [ ]:
# ─── 3-A: Build the vector store ──────────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

DOCUMENTS = [
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs. "
                     "It models workflows as a directed graph of nodes and edges.",
        metadata={"source": "langgraph-docs", "topic": "framework"},
    ),
    Document(
        page_content="Self-RAG teaches LLMs to generate reflection tokens that decide when to retrieve. "
                     "The tokens are ISRET (should retrieve?), ISREL (doc relevant?), and ISSUP (answer supported?).",
        metadata={"source": "self-rag-paper", "topic": "rag"},
    ),
    Document(
        page_content="Corrective RAG grades retrieved documents and rewrites the query when docs are "
                     "irrelevant, falling back to a web search if the local store has no good results.",
        metadata={"source": "crag-paper", "topic": "rag"},
    ),
    Document(
        page_content="RAG Fusion generates multiple query variants then merges results via "
                     "Reciprocal Rank Fusion to improve recall across diverse phrasings.",
        metadata={"source": "rag-fusion", "topic": "rag"},
    ),
    Document(
        page_content="The Eiffel Tower is 330 metres tall and stands in Paris, France. "
                     "It was completed in 1889 as the entrance arch for the World Fair.",
        metadata={"source": "general", "topic": "facts"},
    ),
    Document(
        page_content="Adaptive RAG classifies each query and routes it to the cheapest correct "
                     "retrieval path: no retrieval, single-step, or iterative multi-hop retrieval.",
        metadata={"source": "adaptive-rag", "topic": "rag"},
    ),
    Document(
        page_content="Pydantic BaseModel with_structured_output forces an LLM to emit JSON "
                     "conforming to a given schema. Used in Self-RAG to implement boolean gates.",
        metadata={"source": "pydantic-docs", "topic": "framework"},
    ),
]

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
store = Chroma.from_documents(DOCUMENTS, embeddings_model)
retriever = store.as_retriever(search_kwargs={"k": 2})

print(f"Knowledge base ready: {len(DOCUMENTS)} documents")
print("Topics:", list({d.metadata['topic'] for d in DOCUMENTS}))

In [ ]:
# ─── 3-B: Inspect retrieval before wiring gates ───────────────────────────────
# Always verify what the retriever returns before trusting any gate output.

test_queries = [
    "What is Self-RAG?",
    "How tall is the Eiffel Tower?",
    "What is 17 times 6?",
]

for q in test_queries:
    docs = retriever.invoke(q)
    print(f"Q: {q}")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] ({d.metadata['source']}) {d.page_content[:80]}...")
    print()

---

## Part 4 — Gate 1: classify (ISRET — Should I Retrieve?)

---

### The BoolDecision Pattern

All three Self-RAG gates use the **same one-field Pydantic model** as the structured output schema. The LLM is forced to emit `{"answer": true}` or `{"answer": false}` — no free-text, no ambiguity.

```python
class BoolDecision(BaseModel):
    answer: bool

gate = llm.with_structured_output(BoolDecision)
result = gate.invoke("Should I retrieve for: 'What is 2 + 2?'")
# → BoolDecision(answer=False)
```

### Gate 1 Decision Logic

| Query type | Expected ISRET | Reason |
|-----------|----------------|--------|
| Complex knowledge question | `True` | Model needs external grounding |
| Math / arithmetic | `False` | Model computes this from weights |
| Greeting / small talk | `False` | No knowledge needed |
| Recent private knowledge | `True` | Model doesn't know it |
| Simple geography fact | `False` or `True` | Model may know — prompt matters |

In [ ]:
# ─── 4-A: BoolDecision — the shared structured output schema ──────────────────
from langchain_openai import ChatOpenAI
from pydantic import BaseModel


class BoolDecision(BaseModel):
    answer: bool


_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
_gate = _llm.with_structured_output(BoolDecision)

print("BoolDecision schema:")
print(BoolDecision.model_json_schema())

In [ ]:
# ─── 4-B: Test Gate 1 manually ────────────────────────────────────────────────
# Before wiring into the graph, probe each gate in isolation.

GATE1_PROMPT = (
    "Should I search a knowledge base to answer this question?\n"
    "Say FALSE for: math, greetings, or simple facts the model knows well.\n"
    "Say TRUE for: knowledge-base-specific facts, recent events, or complex technical topics.\n"
    "Question: {question}"
)

gate1_tests = [
    "What is Self-RAG?",
    "What is 17 times 6?",
    "Hello, how are you?",
    "How does Adaptive RAG route queries?",
    "What year was the Eiffel Tower built?",
]

print(f"{'Query':<45} {'Retrieve?':>10}")
print("─" * 57)
for q in gate1_tests:
    result = _gate.invoke(GATE1_PROMPT.format(question=q))
    label = "YES — retrieve" if result.answer else "NO  — skip"
    print(f"{q:<45} {label:>15}")

In [ ]:
# ─── 4-C: Effect of prompt wording on Gate 1 ─────────────────────────────────
# Prompt engineering changes the gate's sensitivity significantly.
# Compare a permissive prompt vs. a conservative prompt.

PERMISSIVE = "Should I search a knowledge base to answer: '{q}'? Say true if unsure."
CONSERVATIVE = (
    "Should I search a knowledge base to answer: '{q}'? "
    "Say false unless the model definitely cannot answer from its own weights."
)

boundary_queries = [
    "What is the capital of France?",
    "How does RAG work?",
    "What is the speed of light?",
]

print(f"{'Query':<40} {'Permissive':>12} {'Conservative':>14}")
print("─" * 68)
for q in boundary_queries:
    r1 = _gate.invoke(PERMISSIVE.format(q=q)).answer
    r2 = _gate.invoke(CONSERVATIVE.format(q=q)).answer
    print(f"{q:<40} {str(r1):>12} {str(r2):>14}")

print("\nObservation: prompt framing shifts where the decision boundary falls.")

---

## Part 5 — Gate 2: grade_docs (ISREL — Are These Docs Relevant?)

---

Gate 2 runs **once per retrieved document** and filters out any docs whose cosine similarity happened to be high but whose content doesn't actually answer the question. This is the "corrective" step that prevents noise from polluting the generation context.

### Why filtering matters

| Scenario | Without Gate 2 | With Gate 2 |
|----------|---------------|-------------|
| Retriever returns 2 relevant + 1 off-topic | All 3 go to LLM — noise dilutes answer | Only 2 go to LLM |
| All retrieved docs are off-topic | LLM may hallucinate or blend noise | 0 docs pass — model uses its weights |
| All retrieved docs are relevant | No change | All pass through |

In [ ]:
# ─── 5-A: Test Gate 2 manually — per-document relevance grading ───────────────

GATE2_PROMPT = (
    "Is this document relevant to answering the question?\n"
    "Say true only if the document contains information that directly helps answer it.\n"
    "Question: {question}\n"
    "Document: {doc}"
)

question = "What is Self-RAG and how does it differ from standard RAG?"
candidates = retriever.invoke(question)

print(f"Query: '{question}'")
print(f"Retrieved {len(candidates)} docs — grading each:\n")

for doc in candidates:
    result = _gate.invoke(GATE2_PROMPT.format(question=question, doc=doc.page_content))
    label = "RELEVANT" if result.answer else "FILTERED OUT"
    print(f"  [{label}] ({doc.metadata['source']})")
    print(f"    {doc.page_content[:100]}...")
    print()

In [ ]:
# ─── 5-B: Force a filtering scenario — inject an off-topic document ───────────
# Verify Gate 2 removes noise even when the retriever returns it.

off_topic_doc = Document(
    page_content="The best pasta recipe uses 00 flour, eggs, and a pinch of salt. Rest the dough for 30 minutes.",
    metadata={"source": "recipe-blog", "topic": "cooking"},
)

mixed_candidates = [candidates[0], off_topic_doc]
question2 = "How does LangGraph represent workflows?"

print(f"Query: '{question2}'")
print("Candidates (1 relevant + 1 off-topic):\n")

kept = []
for doc in mixed_candidates:
    result = _gate.invoke(GATE2_PROMPT.format(question=question2, doc=doc.page_content))
    label = "KEPT" if result.answer else "DROPPED"
    if result.answer:
        kept.append(doc)
    print(f"  [{label}] ({doc.metadata['source']}) {doc.page_content[:80]}...")

print(f"\nDocs passed to generator: {len(kept)} / {len(mixed_candidates)}")

---

## Part 6 — Gate 3: check_support (ISSUP — Is the Answer Grounded?)

---

Gate 3 runs **after generation** and asks: does the produced answer actually follow from the retrieved context? This catches cases where the LLM blended retrieved content with its own training data and produced a claim not found in the context.

### Grading criteria

| Scenario | ISSUP result | Meaning |
|----------|-------------|----------|
| Answer is a direct paraphrase of context | `True` | Fully grounded |
| Answer adds facts from model weights | `False` | Partially hallucinated |
| No context was retrieved (math / greeting) | `True` (bypass) | Nothing to verify |
| Context returned 0 docs after Gate 2 | `True` (bypass) | Model used weights — still valid |

In [ ]:
# ─── 6-A: Test Gate 3 manually — is the answer supported? ────────────────────
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

GATE3_PROMPT = (
    "Is this answer fully supported by the context?\n"
    "Say true only if every claim in the answer can be found in or directly inferred from the context.\n"
    "Context: {ctx}\n"
    "Answer: {answer}"
)

ctx_text = "Self-RAG teaches LLMs to generate reflection tokens that decide when to retrieve."

grounded_answer = "Self-RAG uses reflection tokens to control when the model retrieves information."
hallucinated_answer = (
    "Self-RAG was developed at Stanford in 2022 and has 7 different token types "
    "that control retrieval, ranking, and summarization."
)

r_grounded = _gate.invoke(GATE3_PROMPT.format(ctx=ctx_text, answer=grounded_answer))
r_hallucinated = _gate.invoke(GATE3_PROMPT.format(ctx=ctx_text, answer=hallucinated_answer))

print(f"Context: '{ctx_text}'\n")
print(f"Grounded answer     → ISSUP = {r_grounded.answer}")
print(f"  '{grounded_answer}'")
print()
print(f"Hallucinated answer → ISSUP = {r_hallucinated.answer}")
print(f"  '{hallucinated_answer[:100]}...'")

In [ ]:
# ─── 6-B: Gate 3 bypass when no context ──────────────────────────────────────
# When Gate 1 skipped retrieval (no context), Gate 3 automatically passes.

math_question = "What is 17 times 6?"
no_context = []

gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using the context if provided, otherwise use your knowledge."),
    ("human", "Context:\n{ctx}\n\nQuestion: {question}"),
])
answer = (gen_prompt | _llm | StrOutputParser()).invoke({
    "ctx": "(no retrieved documents)",
    "question": math_question,
})

# Gate 3 bypass logic — mirrors what check_support does in the graph
if not no_context:
    supported = True
    reason = "bypassed — no documents retrieved"
else:
    ctx = "\n\n".join(d.page_content for d in no_context)
    supported = _gate.invoke(GATE3_PROMPT.format(ctx=ctx, answer=answer)).answer
    reason = "graded against context"

print(f"Q: {math_question}")
print(f"A: {answer}")
print(f"Supported: {supported} ({reason})")

---

## Part 7 — Wire All Three Gates into a LangGraph Workflow

---

### State and Nodes

LangGraph workflows carry a `TypedDict` state through each node. Each node receives the full state and returns a partial update (only the keys it changes).

| Node | Reads from state | Writes to state |
|------|-----------------|------------------|
| `classify` | `question` | `should_retrieve` |
| `retrieve` | `question` | `documents` |
| `grade_docs` | `question`, `documents` | `documents` (filtered) |
| `generate` | `question`, `documents` | `generation` |
| `check_support` | `documents`, `generation` | `supported` |

### Conditional Routing

The branch after `classify` uses a **conditional edge** — a Python lambda that inspects the state and returns the name of the next node:

```python
g.add_conditional_edges(
    "classify",
    lambda s: "retrieve" if s.get("should_retrieve") else "generate"
)
```

In [ ]:
# ─── 7-A: Define the graph state and all five nodes ───────────────────────────
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    question: str
    documents: list
    generation: str
    should_retrieve: bool
    supported: bool


# ── Gate 1 ────────────────────────────────────────────────────────────────────
def classify(state: State) -> dict:
    """ISRET — should we retrieve at all?"""
    result = _gate.invoke(
        f"Should I search a knowledge base to answer this question?\n"
        f"Say false for math, greetings, or simple facts the model knows well.\n"
        f"Question: {state['question']}"
    )
    return {"should_retrieve": result.answer}


# ── Retrieval ─────────────────────────────────────────────────────────────────
def retrieve(state: State) -> dict:
    return {"documents": retriever.invoke(state["question"])}


# ── Gate 2 ────────────────────────────────────────────────────────────────────
def grade_docs(state: State) -> dict:
    """ISREL — keep only documents relevant to the question."""
    relevant = [
        d for d in state["documents"]
        if _gate.invoke(
            f"Is this document relevant to answering the question?\n"
            f"Question: {state['question']}\nDocument: {d.page_content}"
        ).answer
    ]
    return {"documents": relevant}


# ── Generation ────────────────────────────────────────────────────────────────
def generate(state: State) -> dict:
    ctx = "\n\n".join(d.page_content for d in state.get("documents", []))
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using the context if provided, otherwise use your knowledge."),
        ("human", "Context:\n{ctx}\n\nQuestion: {question}"),
    ])
    answer = (prompt | _llm | StrOutputParser()).invoke({
        "ctx": ctx or "(no retrieved documents)",
        "question": state["question"],
    })
    return {"generation": answer}


# ── Gate 3 ────────────────────────────────────────────────────────────────────
def check_support(state: State) -> dict:
    """ISSUP — is the answer grounded in the retrieved context?"""
    if not state.get("documents"):
        return {"supported": True}  # bypass: nothing to verify against
    ctx = "\n\n".join(d.page_content for d in state["documents"])
    result = _gate.invoke(
        f"Is this answer fully supported by the context?\n"
        f"Context: {ctx}\nAnswer: {state['generation']}"
    )
    return {"supported": result.answer}


print("All five nodes defined: classify, retrieve, grade_docs, generate, check_support")

In [ ]:
# ─── 7-B: Assemble and compile the graph ─────────────────────────────────────

g = StateGraph(State)

# Add all five nodes
g.add_node("classify", classify)
g.add_node("retrieve", retrieve)
g.add_node("grade_docs", grade_docs)
g.add_node("generate", generate)
g.add_node("check_support", check_support)

# Entry point
g.add_edge(START, "classify")

# Gate 1 conditional branch
g.add_conditional_edges(
    "classify",
    lambda s: "retrieve" if s.get("should_retrieve") else "generate",
)

# Fixed edges
g.add_edge("retrieve", "grade_docs")
g.add_edge("grade_docs", "generate")
g.add_edge("generate", "check_support")
g.add_edge("check_support", END)

app = g.compile()
print("Graph compiled successfully.")
print("Flow: START → classify → [retrieve → grade_docs →] generate → check_support → END")

---

## Part 8 — Run the Full Self-RAG Pipeline

---

We run four question types to observe all code paths:

| Question | Expected path |
|----------|---------------|
| "What is Self-RAG?" | classify → retrieve → grade_docs → generate → check_support |
| "What is 17 times 6?" | classify → generate → check_support (bypassed) |
| "How does Adaptive RAG work?" | classify → retrieve → grade_docs → generate → check_support |
| "Hello, how are you?" | classify → generate → check_support (bypassed) |

In [ ]:
# ─── 8-A: Run all four test cases and print a trace ──────────────────────────

INITIAL_STATE = {
    "question": "",
    "documents": [],
    "generation": "",
    "should_retrieve": False,
    "supported": False,
}

QUESTIONS = [
    "What is Self-RAG?",
    "What is 17 times 6?",
    "How does Adaptive RAG work?",
    "Hello, how are you?",
]

for q in QUESTIONS:
    result = app.invoke({**INITIAL_STATE, "question": q})
    retrieved = result["should_retrieve"]
    n_docs = len(result["documents"])
    supported = result["supported"]

    path = (
        f"classify({'RETRIEVE' if retrieved else 'SKIP'}) → "
        + (f"retrieve({n_docs} docs) → grade_docs → " if retrieved else "")
        + f"generate → check_support({'PASS' if supported else 'FAIL'})"
    )

    print(f"Q: {q}")
    print(f"   Path:    {path}")
    print(f"   Answer:  {result['generation'][:150]}")
    print()

In [ ]:
# ─── 8-B: Step-through trace — see every node fire ───────────────────────────
# stream_mode="values" emits the full state after each node.
# This lets you trace exactly what each gate changed.

trace_question = "What is Self-RAG?"

print(f"Step-through trace for: '{trace_question}'\n")
print("─" * 60)

prev_keys = {}
for step_state in app.stream(
    {**INITIAL_STATE, "question": trace_question},
    stream_mode="values",
):
    # Find what changed vs. previous step
    changed = {k: v for k, v in step_state.items() if prev_keys.get(k) != v}
    if changed:
        for k, v in changed.items():
            if k == "documents":
                print(f"  {k}: [{len(v)} doc(s)] {[d.metadata['source'] for d in v]}")
            elif k == "generation":
                print(f"  {k}: {str(v)[:120]}...")
            else:
                print(f"  {k}: {v}")
        print("─" * 60)
    prev_keys = dict(step_state)

---

## Exercises

---

### Exercise 1 — Probe Gate 1 on the Boundary

Ask `"What is the capital of France?"`. The model knows Paris from its weights.
- Does Gate 1 return `False` (skip retrieval) or `True` (retrieve)?
- Now reword the prompt to `"According to our knowledge base, what is the capital of France?"` — does the answer change?
- What does this tell you about how prompt phrasing affects gate sensitivity?

---

### Exercise 2 — Make Gate 2 Filter

Add an off-topic document to the store (e.g., a recipe or a sports fact).
Then rebuild the retriever with `k=3` and ask a LangGraph-related question.
Verify that `grade_docs` drops the off-topic document before generation.

---

### Exercise 3 — Add a Retry Loop

When `check_support` returns `False` (answer not grounded), the current graph simply ends with a bad answer.
Design (or implement) a retry path:
- Route from `check_support` back to `retrieve` with a counter in state
- Cap retries at 2 to avoid infinite loops
- What guard would you add to state? What would the conditional edge lambda look like?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# Probe the gate boundary for a well-known fact.

boundary_q = "What is the capital of France?"

# TODO: invoke Gate 1 with the standard prompt — what does it return?
result_standard = _gate.invoke(GATE1_PROMPT.format(question=boundary_q))
print(f"Standard prompt → retrieve = {result_standard.answer}")

# TODO: reword the question to suggest it should check the knowledge base
# varied_q = "According to our knowledge base, what is the capital of France?"
# result_varied = _gate.invoke(GATE1_PROMPT.format(question=varied_q))
# print(f"KB-phrased prompt → retrieve = {result_varied.answer}")

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
# Force Gate 2 to filter by injecting a noisy document.

# TODO: create a noisy off-topic document
# noisy_doc = Document(
#     page_content="...",
#     metadata={"source": "noise", "topic": "unrelated"},
# )

# TODO: rebuild the store with the noisy doc included and k=3
# noisy_store = Chroma.from_documents(DOCUMENTS + [noisy_doc], embeddings_model)
# noisy_retriever = noisy_store.as_retriever(search_kwargs={"k": 3})

# TODO: run a LangGraph question and check that grade_docs removes the noise
print("Implement your noise-injection experiment here.")

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
# Design a retry loop that re-routes from check_support back to retrieve.

# Hint: add a 'retry_count' field to State and guard with a max-retry check.

# class StateWithRetry(TypedDict):
#     question: str
#     documents: list
#     generation: str
#     should_retrieve: bool
#     supported: bool
#     retry_count: int   # NEW
#
# def retry_or_end(state):
#     if not state["supported"] and state.get("retry_count", 0) < 2:
#         return "retrieve"   # try again
#     return END
#
# g.add_conditional_edges("check_support", retry_or_end)

print("Sketch your retry graph here — uncomment and complete the code above.")

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

---

### Exercise 1 — Probe Gate 1 on the Boundary

**Expected result:** With the standard prompt, `"What is the capital of France?"` usually returns `False` — the LLM knows Paris from its weights and correctly skips retrieval. Re-phrasing to `"According to our knowledge base..."` often flips the gate to `True` because the phrasing explicitly implies an external source is available.

**Key insight:** Gate 1 is a soft heuristic — its boundary is prompt-sensitive. For production, run a small validation set of 50-100 questions and tune the prompt until the false-positive rate (unnecessary retrievals) and false-negative rate (missed retrievals) both fall below your threshold.

---

### Exercise 2 — Make Gate 2 Filter

**Expected result:** With k=3, the noisy document gets retrieved based on vector similarity. Gate 2 correctly labels it `False` (irrelevant) and removes it before generation. The generator receives only the relevant docs.

**Key insight:** Cosine similarity is a noisy signal — it measures textual proximity, not logical relevance. Gate 2 is the semantic filter that corrects the vector store's imprecision.

---

### Exercise 3 — Add a Retry Loop

**Key invariants to enforce:**
- `retry_count` starts at 0 in the initial state
- The retrieve node increments `retry_count` on each pass
- The guard `retry_count < 2` prevents an infinite loop
- After 2 retries, the graph exits with `supported = False` — caller must handle this

In [ ]:
# ── Answer Key: Exercise 1 ────────────────────────────────────────────────────

boundary_q = "What is the capital of France?"
varied_q = "According to our knowledge base, what is the capital of France?"

r1 = _gate.invoke(GATE1_PROMPT.format(question=boundary_q))
r2 = _gate.invoke(GATE1_PROMPT.format(question=varied_q))

print(f"Standard phrasing     → retrieve = {r1.answer}")
print(f"KB-referencing phrase → retrieve = {r2.answer}")
print("\nConclusion: prompt phrasing shifts the decision boundary.")
print("For production, calibrate with a held-out validation set.")

In [ ]:
# ── Answer Key: Exercise 2 ────────────────────────────────────────────────────

noisy_doc = Document(
    page_content="The best sourdough uses 70% hydration dough and cold ferments overnight in the fridge.",
    metadata={"source": "recipe-blog", "topic": "cooking"},
)

noisy_store = Chroma.from_documents(DOCUMENTS + [noisy_doc], embeddings_model)
noisy_retriever = noisy_store.as_retriever(search_kwargs={"k": 3})

noisy_q = "How does LangGraph model workflows?"
candidates = noisy_retriever.invoke(noisy_q)

print(f"Retrieved {len(candidates)} docs (k=3):")
kept = []
for doc in candidates:
    is_rel = _gate.invoke(
        f"Is this document relevant to: '{noisy_q}'?\nDocument: {doc.page_content}"
    ).answer
    label = "KEPT" if is_rel else "FILTERED"
    if is_rel:
        kept.append(doc)
    print(f"  [{label}] ({doc.metadata['source']}) {doc.page_content[:70]}...")

print(f"\nPassed Gate 2: {len(kept)} / {len(candidates)} documents")

In [ ]:
# ── Answer Key: Exercise 3 — Retry loop ───────────────────────────────────────

from typing import TypedDict


class StateWithRetry(TypedDict):
    question: str
    documents: list
    generation: str
    should_retrieve: bool
    supported: bool
    retry_count: int


def retry_or_end(state: StateWithRetry) -> str:
    """Route: retry retrieval if unsupported and under retry cap."""
    if not state["supported"] and state.get("retry_count", 0) < 2:
        return "retrieve"  # try again
    return END


def retrieve_with_counter(state: StateWithRetry) -> dict:
    """Retrieve and increment the retry counter."""
    docs = retriever.invoke(state["question"])
    return {"documents": docs, "retry_count": state.get("retry_count", 0) + 1}


gr = StateGraph(StateWithRetry)
gr.add_node("classify", classify)
gr.add_node("retrieve", retrieve_with_counter)
gr.add_node("grade_docs", grade_docs)
gr.add_node("generate", generate)
gr.add_node("check_support", check_support)

gr.add_edge(START, "classify")
gr.add_conditional_edges(
    "classify",
    lambda s: "retrieve" if s.get("should_retrieve") else "generate",
)
gr.add_edge("retrieve", "grade_docs")
gr.add_edge("grade_docs", "generate")
gr.add_edge("generate", "check_support")
gr.add_conditional_edges("check_support", retry_or_end)

retry_app = gr.compile()

print("Retry graph compiled.")
print("check_support → retry_or_end → 'retrieve' (max 2x) or END")
print()
print("Key invariants:")
print("  - retry_count starts at 0 in initial state")
print("  - retrieve_with_counter increments it each time")
print("  - guard: retry_count < 2 prevents infinite loop")

---

## What's Next?

You now understand Self-RAG's three-gate architecture and how to wire structured LLM decisions as conditional graph edges. Here's where to go deeper:

### Related examples in this repo
- **17-corrective-rag** — corrects *after* retrieval: grade docs → rewrite query → web search fallback
- **25-adaptive-rag** — routes queries to different retrieval strategies *before* fetching
- **29-llm-judge-harness** — use structured output to score RAG answers on a 0-3 rubric (ISUSE extended)
- **30-agentic-rag** — full agentic loop where the LLM decides how many retrieval iterations to run

### Extend this implementation
- **Add ISUSE (utility) gate** — after `check_support`, rate how useful the answer is (0-3 Likert scale via `with_structured_output`)
- **Add query rewriting** — when `grade_docs` drops all docs, rewrite the query before a second retrieval pass
- **Add LangSmith tracing** — `os.environ["LANGCHAIN_TRACING_V2"] = "true"` traces every gate call in a dashboard
- **Scale the knowledge base** — replace the in-memory store with a persistent ChromaDB collection or a Supabase `pgvector` table
- **Fine-tune the classifiers** — the gate prompts are heuristics; collect labeled examples and fine-tune a small model for each gate

### Further reading
> Asai, A., Wu, Z., Wang, Y., Sil, A., & Hajishirzi, H. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection.* https://arxiv.org/abs/2310.11511
> Yan, S., et al. (2024). *Corrective Retrieval Augmented Generation.* https://arxiv.org/abs/2401.15884
> Jeong, S., et al. (2024). *Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models.* https://arxiv.org/abs/2403.14403
> LangGraph conceptual guide: https://langchain-ai.github.io/langgraph/concepts/

---

*Workshop complete. The next step is example 17 (Corrective RAG) — the same gate pattern with a web-search fallback when all local docs fail.*